# Capítulo 6 - Fine-Tuning para Classificação e Instrução

Este notebook demonstra as principais técnicas do capítulo 6:

1. **Carregamento de pesos pré-treinados** do GPT-2 (HuggingFace → nosso GPTModel)  
2. **Fine-tuning de classificação** — detecção de spam (SMS Spam Collection)  
3. **Avaliação** — acurácia, F1, precisão e recall  
4. **Instruction fine-tuning** — formato Alpaca com mascaramento de prompt  

> **Referência:** Build a Large Language Model (from Scratch) — Sebastian Raschka, cap. 6

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import tiktoken

from src.model.config import GPT_CONFIG_124M, criar_config

# GPT-2 usa QKV bias — necessário para carregar pesos pré-treinados
CFG_GPT2 = dict(GPT_CONFIG_124M)
CFG_GPT2["qkv_bias"] = True

tokenizer = tiktoken.get_encoding("gpt2")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Config GPT-2 124M: {CFG_GPT2['n_layers']} blocos, {CFG_GPT2['emb_dim']} dims")

## 6.1 Carregando Pesos Pré-treinados do GPT-2

O GPT-2 original (OpenAI) foi treinado em 40 GB de texto. Ao invés de pré-treinar do zero,
carregamos esses pesos no nosso `GPTModel`.

**Desafio principal:** A HuggingFace usa `Conv1D` onde usamos `nn.Linear` —
os pesos ficam transpostos e precisam ser corrigidos com `.T`.

In [ ]:
from src.finetuning.pesos_pretrained import baixar_gpt2_hf, carregar_pesos_gpt2, verificar_mapeamento
from src.model.gpt_model import GPTModel

# Carrega modelo pré-treinado da HuggingFace
# (requer: pip install transformers; internet na primeira execução)
print("Baixando GPT-2 da HuggingFace...")
gpt2_hf = baixar_gpt2_hf(modelo="gpt2")

# Instancia nosso modelo e copia os pesos
modelo_pretrained = GPTModel(CFG_GPT2)
carregar_pesos_gpt2(modelo_pretrained, gpt2_hf)
print("Pesos carregados com sucesso!")

In [ ]:
# Verifica que os pesos foram copiados corretamente
erros = verificar_mapeamento(modelo_pretrained, gpt2_hf, tolerancia=1e-5)
print(f"Verificação: {len(erros)} erro(s) encontrado(s)")
if erros:
    for e in erros[:5]:
        print(f"  ⚠ {e}")
else:
    print("✓ Todos os pesos coincidem dentro da tolerância")

In [ ]:
# Teste rápido: geração com pesos pré-treinados
from src.model.generation import gerar_texto_simples

modelo_pretrained.eval()
prompt = "Every effort moves you"
tokens = tokenizer.encode(prompt)
entrada = torch.tensor(tokens).unsqueeze(0)

with torch.no_grad():
    saida = gerar_texto_simples(modelo_pretrained, entrada, max_new_tokens=20, context_size=CFG_GPT2["context_length"])

print(f"Prompt:    {prompt!r}")
print(f"Gerado:    {tokenizer.decode(saida[0].tolist())!r}")

## 6.2 Fine-Tuning para Classificação de Spam

**Dataset:** SMS Spam Collection (UCI) — 5.572 mensagens, ~87% ham, ~13% spam.  
Balanceamos para 747 de cada classe e dividimos 70/10/20 (treino/val/teste).

**Estratégia de congelamento:**
- Congela todos os parâmetros do GPT
- Descongela apenas o último bloco Transformer + LayerNorm final
- Adiciona `classification_head` linear (emb_dim → 2)

In [ ]:
from src.finetuning.dados_cls import (
    baixar_dataset_spam,
    carregar_spam_df,
    balancear_classes,
    dividir_treino_val_teste,
    criar_loaders_spam,
    SpamDataset,
)

# Carrega e pré-processa dataset
caminho = baixar_dataset_spam(PROJECT_ROOT / "data")
dados_brutos = carregar_spam_df(caminho)
dados = balancear_classes(dados_brutos)
treino, val, teste = dividir_treino_val_teste(dados)

print(f"Total balanceado: {len(dados)} amostras")
print(f"Treino: {len(treino)} | Val: {len(val)} | Teste: {len(teste)}")
print(f"Exemplo: {treino[0]}")

In [ ]:
# Cria DataLoaders
MAX_LENGTH = 128
BATCH_SIZE = 8

train_loader, val_loader, test_loader = criar_loaders_spam(
    treino, val, teste, tokenizer, max_length=MAX_LENGTH, batch_size=BATCH_SIZE
)

ids, rotulos = next(iter(train_loader))
print(f"Batch input_ids shape: {ids.shape}")
print(f"Batch labels shape:    {rotulos.shape}")
print(f"Labels (primeiros 8):  {rotulos.tolist()}")

In [ ]:
from src.finetuning.classificador import GPTClassifier

# Instancia classificador com pesos pré-treinados
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

modelo_cls = GPTClassifier(CFG_GPT2, num_classes=2, n_blocos_treinaveis=1)
# Copia pesos pré-treinados (exceto out_head, que foi substituído)
carregar_pesos_gpt2(modelo_cls.gpt, gpt2_hf)
modelo_cls = modelo_cls.to(device)

treinaveis, total = modelo_cls.contar_parametros_treinaveis()
print(f"Parâmetros treináveis: {treinaveis:,} / {total:,} ({100*treinaveis/total:.1f}%)")

In [ ]:
from src.finetuning.classificador import treinar_classificador

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, modelo_cls.parameters()),
    lr=5e-5, weight_decay=0.1
)

historico = treinar_classificador(
    modelo_cls, train_loader, val_loader, optimizer, device,
    num_epochs=5, eval_freq=50
)

print(f"\nTreino concluído!")
print(f"Melhor val loss: {min(historico['val_loss']):.4f}")

## 6.3 Avaliação do Classificador

Avaliamos no conjunto de **teste** (nunca visto durante treino ou validação).
Métricas implementadas manualmente (sem sklearn).

In [ ]:
from src.finetuning.classificador import avaliar_classificador

modelo_cls.eval()
metricas = avaliar_classificador(modelo_cls, test_loader, device)

print("=" * 40)
print("Avaliação no Conjunto de Teste")
print("=" * 40)
for k, v in metricas.items():
    print(f"{k:<15}: {v:.4f}")

In [ ]:
# Teste com exemplos reais
exemplos = [
    "Congratulations! You've won a FREE iPhone. Click now!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT: Your account has been compromised. Send your password NOW.",
    "Can you pick up some milk on the way home?",
]

modelo_cls.eval()
for texto in exemplos:
    tokens = tokenizer.encode(texto)[:MAX_LENGTH]
    entrada = torch.tensor(tokens).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = modelo_cls(entrada)
    pred = logits.argmax(dim=-1).item()
    classe = "SPAM 🚨" if pred == 1 else "HAM  ✓"
    print(f"{classe} | {texto[:60]}...")

## 6.4 Instruction Fine-Tuning (Formato Alpaca)

O instruction fine-tuning ensina o modelo a **seguir instruções** em linguagem natural.

**Formato Alpaca:**
```
### Instruction:
<instrução>

### Input:
<contexto opcional>

### Response:
<resposta>
```

**Mascaramento de prompt:** Os tokens da instrução recebem `ignore_index=-100`
para que a cross-entropy só penalize erros na resposta.

In [ ]:
from src.finetuning.instrucoes import formatar_prompt, InstructionDataset, collate_instrucoes

# Demonstra o formato Alpaca
ex_sem_input = {"instruction": "Converta Celsius para Fahrenheit", "input": "", "output": "F = C × 9/5 + 32"}
ex_com_input = {"instruction": "Traduza para o português", "input": "The cat sat on the mat.", "output": "O gato sentou no tapete."}

print("=== Sem input ===")
print(formatar_prompt(ex_sem_input))
print("\n=== Com input ===")
print(formatar_prompt(ex_com_input))

In [ ]:
# Demonstra mascaramento: tokens do prompt → -100
ds_demo = InstructionDataset([ex_com_input], tokenizer, max_length=64)
input_ids, target_ids = ds_demo[0]

n_mascara = (target_ids == InstructionDataset.IGNORE_INDEX).sum().item()
n_resposta = (target_ids != InstructionDataset.IGNORE_INDEX).sum().item()

print(f"Tokens totais:   {len(input_ids)}")
print(f"Tokens mascarados (prompt):  {n_mascara}")
print(f"Tokens não mascarados (resp): {n_resposta}")
print(f"\nPrimeiros target_ids: {target_ids[:20].tolist()}")

In [ ]:
import json, urllib.request
from torch.utils.data import DataLoader
from functools import partial

# Carrega dataset de instruções (1100 exemplos)
URL = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
try:
    with urllib.request.urlopen(URL) as r:
        instrucoes = json.load(r)
    print(f"Dataset carregado: {len(instrucoes)} exemplos")
except Exception as e:
    # Fallback: usa exemplos sintéticos para demo
    print(f"Erro ao baixar ({e}). Usando exemplos sintéticos.")
    instrucoes = [
        {"instruction": f"Instrução {i}", "input": "", "output": f"Resposta {i}"}
        for i in range(50)
    ]

# Split 85/15
split = int(0.85 * len(instrucoes))
treino_inst = instrucoes[:split]
val_inst    = instrucoes[split:]
print(f"Treino: {len(treino_inst)} | Val: {len(val_inst)}")

In [ ]:
collate_fn = partial(collate_instrucoes, pad_id=tokenizer.n_vocab - 1, device=str(device))

ds_treino_inst = InstructionDataset(treino_inst, tokenizer, max_length=512)
ds_val_inst    = InstructionDataset(val_inst, tokenizer, max_length=512)

loader_treino_inst = DataLoader(ds_treino_inst, batch_size=4, shuffle=True,  collate_fn=collate_fn)
loader_val_inst    = DataLoader(ds_val_inst,    batch_size=4, shuffle=False, collate_fn=collate_fn)

ids, tgt = next(iter(loader_treino_inst))
print(f"Batch ids shape:    {ids.shape}")
print(f"Batch target shape: {tgt.shape}")
print(f"Tokens ignorados no target: {(tgt == -100).sum().item()}")

In [ ]:
from src.finetuning.instrucoes import treinar_instrucoes

# Reutiliza o modelo pré-treinado (sem a cabeça de classificação)
modelo_inst = GPTModel(CFG_GPT2).to(device)
carregar_pesos_gpt2(modelo_inst, gpt2_hf)

optimizer_inst = torch.optim.AdamW(modelo_inst.parameters(), lr=5e-5, weight_decay=0.1)

exemplo_prompt = formatar_prompt(val_inst[0])

historico_inst = treinar_instrucoes(
    modelo_inst, loader_treino_inst, loader_val_inst,
    optimizer_inst, device, num_epochs=2,
    eval_freq=50, tokenizer=tokenizer, exemplo_prompt=exemplo_prompt
)

print(f"\nFine-tuning concluído!")
print(f"Val loss final: {historico_inst['val_loss'][-1]:.4f}")

## 6.5 Análise Comparativa

| Técnica | Parâmetros Treináveis | Dados | Objetivo |
|---|---|---|---|
| Classification FT | ~1% (último bloco + head) | SMS Spam 747+747 | Rótulo binário |
| Instruction FT | 100% | 1.100 exemplos Alpaca | Seguir instruções |

**Pontos chave do capítulo:**
- Conv1D vs nn.Linear: pesos transpostos ao importar do HuggingFace
- Congelamento seletivo: acelera treino e evita catastrofic forgetting
- Token `[:, -1, :]`: o último token resume o contexto completo (GPT causal)
- `ignore_index=-100`: a cross-entropy ignora tokens do prompt automaticamente
- Dynamic padding via `collate_fn`: eficiência de memória por batch

In [ ]:
# Resumo dos resultados
print("=" * 50)
print("RESUMO DO CAPÍTULO 6")
print("=" * 50)
print(f"{'Classificação de Spam':}")  
if 'metricas' in dir():
    for k, v in metricas.items():
        print(f"  {k:<15}: {v:.4f}")

print(f"\nInstruction Fine-Tuning")
if 'historico_inst' in dir() and historico_inst['val_loss']:
    print(f"  Val loss inicial: {historico_inst['val_loss'][0]:.4f}")
    print(f"  Val loss final:   {historico_inst['val_loss'][-1]:.4f}")
    melhora = historico_inst['val_loss'][0] - historico_inst['val_loss'][-1]
    print(f"  Melhora:          {melhora:.4f}")